# MoCap Analysis — Transform to TFrame Coordinate System
**Pipeline:**
1. Locate repo root and load script dynamically
2. Read CSV → 3 rigid body DataFrames
3. Add datetime, truncate to shared window
4. Inspect raw data
5. Define and run coordinate frame transform
6. Sanity checks
7. Print noark m5 position in tframe (metres)
8. Print table m1–m5 positions in tframe (metres)

## Cell 1 — Imports

In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

print('Libraries loaded OK')

## Cell 2 — Locate repo root and load `multiple_rigid_body.py`

Uses `Path` to walk up from the notebook's directory until it finds
the repo root (identified by the presence of a `vaideesh/` folder or `.venv/`).
Then loads the script via `importlib` — no `%run` needed.

In [ ]:
# ── locate this notebook's directory ─────────────────────────────────────────
# __file__ is not defined in notebooks; use VS Code's injected var if present,
# otherwise fall back to the current working directory.
try:
    _nb_file = Path(globals()['__vsc_ipynb_file__']).resolve()
    _nb_dir  = _nb_file.parent
except KeyError:
    _nb_dir = Path.cwd()

# ── walk up to find repo root ─────────────────────────────────────────────────
# Repo root is the first ancestor that contains 'vaideesh/' or '.venv/'
repo_root = _nb_dir
for candidate in [_nb_dir, *_nb_dir.parents]:
    if (candidate / 'vaideesh').exists() or (candidate / '.venv').exists():
        repo_root = candidate
        break

print(f'Notebook dir : {_nb_dir}')
print(f'Repo root    : {repo_root}')

# ── load multiple_rigid_body.py via importlib ─────────────────────────────────
_script_path = repo_root / 'vaideesh' / 'Analysis' / 'multiple_rigid_body.py'
assert _script_path.exists(), f'Script not found at: {_script_path}'

spec = importlib.util.spec_from_file_location('multiple_rigid_body', _script_path)
mrb  = importlib.util.module_from_spec(spec)
sys.modules['multiple_rigid_body'] = mrb
spec.loader.exec_module(mrb)

# bring functions into notebook namespace
from multiple_rigid_body import (
    read_3_rigid_body_csv,
    read_markers_from_3rb_csv,
    add_datetime_col_3rb,
    trunkate_3_dfs,
    get_rb_pos_cols,
    get_rb_rot_cols,
    get_rb_marker_name_3rb,
)

print(f'Script loaded : {_script_path}')

## Cell 3 — Read CSV and build DataFrames

In [ ]:
FILE = repo_root / 'mocap_data' / 'table_frame.csv'
assert FILE.exists(), f'CSV not found: {FILE}'

rb_dfs, st_time = read_3_rigid_body_csv(
    str(FILE),
    rb_names=['tframe', 'noark', 'table'],
)

print(f'Capture start time : {st_time}')
for name, df in rb_dfs.items():
    print(f'  {name}: {df.shape}')

## Cell 4 — Add datetime and truncate to shared time window

In [ ]:
rb_dfs = add_datetime_col_3rb(rb_dfs, st_time)

rb_dfs['tframe'], rb_dfs['noark'], rb_dfs['table'] = trunkate_3_dfs(
    rb_dfs['tframe'],
    rb_dfs['noark'],
    rb_dfs['table'],
    display_print=True,
)

print('\nFinal shapes after truncation:')
for name, df in rb_dfs.items():
    print(f'  {name}: {df.shape}')

## Cell 5 — Inspect raw data

In [ ]:
print('── tframe ──')
display(rb_dfs['tframe'].head(3))

In [ ]:
print('── noark ──')
display(rb_dfs['noark'].head(3))

In [ ]:
print('── table ──')
display(rb_dfs['table'].head(3))

In [ ]:
print('NaN rows per body:')
for name, df in rb_dfs.items():
    nan_rows = df.isna().any(axis=1).sum()
    print(f'  {name}: {nan_rows} NaN rows out of {len(df)}')

## Cell 6 — Define `transform_to_tframe()`

**Math per frame:**

| Step | Formula | Notes |
|---|---|---|
| Translate | `Δp = p_world − p_tframe` | remove tframe origin offset |
| Rotate | `p_rel = R_tframe⁻¹ · Δp` | align axes to tframe |
| Orientation | `q_rel = q_tframe⁻¹ ⊗ q_target` | relative rotation |
| Euler | `scipy.Rotation.as_euler('xyz', degrees=True)` | for readability |

**Units:** Motive exports positions in **metres** — no unit conversion needed.

**Output column suffix `_m`** makes it explicit that values are in metres.

In [ ]:
def transform_to_tframe(rb_dfs):
    """
    Transform selected positions and orientations into the tframe coordinate system.

    Positions  : returned in metres (Motive native unit), column suffix _m.
    Quaternions: stored as xyzw (scipy convention).
    Euler angles: xyz convention, degrees.

    Transforms applied
    ------------------
    noark : marker m5 position (m) + rigid body orientation
    table : marker m1-m5 positions (m) + rigid body orientation

    Returns
    -------
    dict with keys 'noark_in_tframe' and 'table_in_tframe'
    """
    tframe_df = rb_dfs['tframe'].copy()
    noark_df  = rb_dfs['noark'].copy()
    table_df  = rb_dfs['table'].copy()

    # ── tframe reference: quaternion (xyzw) and position (metres) ────────────
    q_tf = tframe_df[['tframe_rot_x', 'tframe_rot_y',
                       'tframe_rot_z', 'tframe_rot_w']].values.astype(float)
    p_tf = tframe_df[['tframe_pos_x', 'tframe_pos_y',
                       'tframe_pos_z']].values.astype(float)

    # scipy Rotation — xyzw order matches Motive quaternion export
    rot_tf_inv = R.from_quat(q_tf).inv()   # world → tframe

    # ── helpers ───────────────────────────────────────────────────────────────
    def pos_to_tframe(p_world):
        """(N,3) world positions (m) → (N,3) tframe positions (m)"""
        return rot_tf_inv.apply(p_world - p_tf)

    def rot_to_tframe(q_world_xyzw):
        """(N,4) xyzw world quaternions → scipy Rotation object in tframe"""
        return rot_tf_inv * R.from_quat(q_world_xyzw)

    # ── 1. NOARK — m5 position + rigid body orientation ───────────────────────
    noark_result = pd.DataFrame({
        'frame'  : noark_df['frame'],
        'seconds': noark_df['seconds'],
        'time'   : noark_df['time'],
    })

    # m5 position in tframe (metres)
    p_m5    = noark_df[['noark_marker_m5_x',
                         'noark_marker_m5_y',
                         'noark_marker_m5_z']].values.astype(float)
    p_m5_tf = pos_to_tframe(p_m5)
    noark_result['noark_m5_tf_x_m'] = p_m5_tf[:, 0]
    noark_result['noark_m5_tf_y_m'] = p_m5_tf[:, 1]
    noark_result['noark_m5_tf_z_m'] = p_m5_tf[:, 2]

    # orientation via scipy: quaternion (xyzw) + Euler angles (xyz, degrees)
    q_noark        = noark_df[['noark_rot_x', 'noark_rot_y',
                                'noark_rot_z', 'noark_rot_w']].values.astype(float)
    rot_noark_tf   = rot_to_tframe(q_noark)
    q_noark_tf     = rot_noark_tf.as_quat()                    # (N,4) xyzw
    euler_noark_tf = rot_noark_tf.as_euler('xyz', degrees=True) # (N,3) deg

    noark_result['noark_quat_tf_x']    = q_noark_tf[:, 0]
    noark_result['noark_quat_tf_y']    = q_noark_tf[:, 1]
    noark_result['noark_quat_tf_z']    = q_noark_tf[:, 2]
    noark_result['noark_quat_tf_w']    = q_noark_tf[:, 3]
    noark_result['noark_roll_tf_deg']  = euler_noark_tf[:, 0]
    noark_result['noark_pitch_tf_deg'] = euler_noark_tf[:, 1]
    noark_result['noark_yaw_tf_deg']   = euler_noark_tf[:, 2]

    # ── 2. TABLE — m1–m5 positions + rigid body orientation ───────────────────
    table_result = pd.DataFrame({
        'frame'  : table_df['frame'],
        'seconds': table_df['seconds'],
        'time'   : table_df['time'],
    })

    for i in range(1, 6):
        p_world     = table_df[[f'table_marker_m{i}_x',
                                 f'table_marker_m{i}_y',
                                 f'table_marker_m{i}_z']].values.astype(float)
        p_tf_coords = pos_to_tframe(p_world)
        table_result[f'table_m{i}_tf_x_m'] = p_tf_coords[:, 0]
        table_result[f'table_m{i}_tf_y_m'] = p_tf_coords[:, 1]
        table_result[f'table_m{i}_tf_z_m'] = p_tf_coords[:, 2]

    # orientation via scipy: quaternion (xyzw) + Euler angles (xyz, degrees)
    q_table        = table_df[['table_rot_x', 'table_rot_y',
                                'table_rot_z', 'table_rot_w']].values.astype(float)
    rot_table_tf   = rot_to_tframe(q_table)
    q_table_tf     = rot_table_tf.as_quat()
    euler_table_tf = rot_table_tf.as_euler('xyz', degrees=True)

    table_result['table_quat_tf_x']    = q_table_tf[:, 0]
    table_result['table_quat_tf_y']    = q_table_tf[:, 1]
    table_result['table_quat_tf_z']    = q_table_tf[:, 2]
    table_result['table_quat_tf_w']    = q_table_tf[:, 3]
    table_result['table_roll_tf_deg']  = euler_table_tf[:, 0]
    table_result['table_pitch_tf_deg'] = euler_table_tf[:, 1]
    table_result['table_yaw_tf_deg']   = euler_table_tf[:, 2]

    return {
        'noark_in_tframe': noark_result.reset_index(drop=True),
        'table_in_tframe': table_result.reset_index(drop=True),
    }

print('transform_to_tframe() defined OK')

## Cell 7 — Run the transformation

In [ ]:
tframe_dfs = transform_to_tframe(rb_dfs)

print('noark_in_tframe shape :', tframe_dfs['noark_in_tframe'].shape)
print('table_in_tframe shape :', tframe_dfs['table_in_tframe'].shape)
print('\nnoark_in_tframe columns:')
print(tframe_dfs['noark_in_tframe'].columns.tolist())
print('\ntable_in_tframe columns:')
print(tframe_dfs['table_in_tframe'].columns.tolist())

## Cell 8 — Inspect transformed DataFrames

In [ ]:
print('── noark in tframe ──')
display(tframe_dfs['noark_in_tframe'].head())

In [ ]:
print('── table in tframe ──')
display(tframe_dfs['table_in_tframe'].head())

## Cell 9 — Sanity checks

In [ ]:
# Check 1: tframe position transformed into itself → should be (0, 0, 0) m
p_tf = rb_dfs['tframe'][['tframe_pos_x','tframe_pos_y','tframe_pos_z']].values.astype(float)
q_tf = rb_dfs['tframe'][['tframe_rot_x','tframe_rot_y','tframe_rot_z','tframe_rot_w']].values.astype(float)
rot_tf_inv  = R.from_quat(q_tf).inv()
self_pos    = rot_tf_inv.apply(p_tf - p_tf)
max_pos_err = np.nanmax(np.abs(self_pos))
print(f'Check 1 — self-position = 0 m     : max error = {max_pos_err:.2e} m  →  {"PASS ✓" if max_pos_err < 1e-10 else "FAIL ✗"}')

In [ ]:
# Check 2: tframe orientation relative to itself → identity quaternion (0,0,0,1)
self_rot    = (rot_tf_inv * R.from_quat(q_tf)).as_quat()
identity    = np.array([0., 0., 0., 1.])
max_rot_err = np.nanmax(np.abs(self_rot - identity))
print(f'Check 2 — self-rotation = identity : max deviation = {max_rot_err:.2e}  →  {"PASS ✓" if max_rot_err < 1e-6 else "FAIL ✗"}')
print(f'          row 0 quaternion          = {self_rot[0].round(6)}')

In [ ]:
# Check 3: all output quaternion norms should be ~1.0
for label, xyzw_cols in [
    ('noark', ['noark_quat_tf_x','noark_quat_tf_y','noark_quat_tf_z','noark_quat_tf_w']),
    ('table', ['table_quat_tf_x','table_quat_tf_y','table_quat_tf_z','table_quat_tf_w']),
]:
    q       = tframe_dfs[f'{label}_in_tframe'][xyzw_cols].values.astype(float)
    norms   = np.linalg.norm(q, axis=1)
    valid   = ~np.isnan(norms)
    max_dev = np.max(np.abs(norms[valid] - 1.0))
    print(f'Check 3 — {label} quat norms ≈ 1.0 : max deviation = {max_dev:.2e}  →  {"PASS ✓" if max_dev < 1e-5 else "FAIL ✗"}')

## Cell 10 — noark m5 position in tframe (metres)

In [ ]:
df_n = tframe_dfs['noark_in_tframe']

print('noark Marker 5 — position in tframe coordinate system (metres)')
print('=' * 64)
print(f'{"Frame":>6}  {"Time (s)":>9}  {"X (m)":>12}  {"Y (m)":>12}  {"Z (m)":>12}')
print('-' * 64)
for _, row in df_n[['frame','seconds',
                     'noark_m5_tf_x_m',
                     'noark_m5_tf_y_m',
                     'noark_m5_tf_z_m']].iterrows():
    fmt = lambda v: f'{v:12.6f}' if not np.isnan(v) else f'{"NaN":>12}'
    print(f"{int(row['frame']):>6}  {row['seconds']:>9.3f}  "
          f"{fmt(row['noark_m5_tf_x_m'])}  "
          f"{fmt(row['noark_m5_tf_y_m'])}  "
          f"{fmt(row['noark_m5_tf_z_m'])}")

print('=' * 64)
print('\nSummary statistics (metres):')
display(
    df_n[['noark_m5_tf_x_m','noark_m5_tf_y_m','noark_m5_tf_z_m']]
    .describe().round(6)
)

## Cell 11 — Table marker m1–m5 positions in tframe (metres)

In [ ]:
df_t = tframe_dfs['table_in_tframe']

for m_idx in range(1, 6):
    xc = f'table_m{m_idx}_tf_x_m'
    yc = f'table_m{m_idx}_tf_y_m'
    zc = f'table_m{m_idx}_tf_z_m'

    print(f'table Marker {m_idx} — position in tframe coordinate system (metres)')
    print('=' * 64)
    print(f'{"Frame":>6}  {"Time (s)":>9}  {"X (m)":>12}  {"Y (m)":>12}  {"Z (m)":>12}')
    print('-' * 64)
    for _, row in df_t[['frame','seconds', xc, yc, zc]].iterrows():
        fmt = lambda v: f'{v:12.6f}' if not np.isnan(v) else f'{"NaN":>12}'
        print(f"{int(row['frame']):>6}  {row['seconds']:>9.3f}  "
              f"{fmt(row[xc])}  {fmt(row[yc])}  {fmt(row[zc])}")
    print('=' * 64)
    print(f'Summary statistics — Marker {m_idx} (metres):')
    display(df_t[[xc, yc, zc]].describe().round(6))
    print()